# Thêm Thư Viện

In [7]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [8]:
conn_libol = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=libol;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2024;'
)
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=dwh_library;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2024;'
)

## Đọc data từ SQL Server

In [9]:
query_Nhomnghanhnghe = "SELECT ID, dbo.DecodeUTF8String(Ten_nhom) AS Ten_nhom FROM Nhom_nghanh_nghe"
df_nhomnghanhnge = pd.read_sql(query_Nhomnghanhnghe, conn_libol)
print(df_nhomnghanhnge)

C:\Users\admin\AppData\Local\Temp\ipykernel_14656\173999304.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_nhomnghanhnge = pd.read_sql(query_Nhomnghanhnghe, conn_libol)


      ID                  Ten_nhom
0      0          (Không xác định)
1      1                      Y tế
2      5       Công nhân viên chức
3      9         Điện tử - Tin học
4     11                 Tài chính
..   ...                       ...
248  291        Công nghệ vật liệu
249  292         Vật liệu xây dựng
250  293                      Luật
251  294         Sư phạm công nghệ
252  295  Kỹ thuật cơ khí động lực

[253 rows x 2 columns]


## Xử lý data

In [10]:
for j, row in df_nhomnghanhnge.iterrows(): 
    ten_nhom = row["Ten_nhom"]
    if pd.isna(ten_nhom) or ten_nhom == "":  # Kiểm tra none hoặc NaN
        df_nhomnghanhnge.at[j, 'Ten_nhom'] = "(Không xác định)" 
df_nhomnghanhnge = df_nhomnghanhnge.drop_duplicates(subset='Ten_nhom').reset_index(drop=True) # xóa những hàng bị trùng nhau
print(df_nhomnghanhnge)

      ID                          Ten_nhom
0      0                  (Không xác định)
1      1                              Y tế
2      5               Công nhân viên chức
3      9                 Điện tử - Tin học
4     11                         Tài chính
..   ...                               ...
232  288  Logistic và Tài chính thương mại
233  292                 Vật liệu xây dựng
234  293                              Luật
235  294                 Sư phạm công nghệ
236  295          Kỹ thuật cơ khí động lực

[237 rows x 2 columns]


## Load data

### [Nếu cần] Clear bảng

In [11]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM DIM_Nhom_nghanh_nghe"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load vào bảng DIM_Nhom_nghanh_nghe

In [12]:
cursor_dwh = conn_dwh_library.cursor()
insert_query = """
                INSERT INTO DIM_Nhom_nghanh_nghe (ID_nhom_nghanh_nghe, 
                            Nhom_nghanh_nghe) 
                VALUES (?, ?)
                """
for index, row in df_nhomnghanhnge.iterrows():
    values = (row['ID'], 
              row['Ten_nhom'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_library.commit()